<a href="https://colab.research.google.com/github/jenithapatel/deeplearning-for-histone-mods/blob/main/notebook_03_5model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing 5 Models for DNA Sequence Classification

**Task:** TF-binding-style prediction — given a stretch of DNA, predict whether a specific regulatory protein binds to it.

**Models compared:**
1. **GLM (nanoGPT)** — character-level transformer language model, pretrained on the human genome and then fine-tuned for classification *(same architecture as the working notebook)*.
2. **1D CNN (Basset / DeepBind style)** — convolutional network over one-hot DNA.
3. **Pure BiLSTM (RNN)** — bidirectional recurrent network over embedded nucleotides, no convolutions.
4. **CNN + BiLSTM (DanQ style)** — convolution for motifs, recurrence for long-range context.
5. **XGBoost on k-mer features** — classical gradient-boosted trees over k-mer counts.

All five train on the **exact same train / validation / test split**, so the numbers are directly comparable.

## A 60-second primer for non-biologists

**DNA is a string.** Every cell carries a roughly 3-billion-character document written in a 4-letter alphabet: `A`, `C`, `G`, `T`. Each letter is a *nucleotide*. The order of these letters encodes everything from your eye color to whether a cell becomes a neuron or a muscle cell.

**Genes are short substrings of this document — but they don't run themselves.** Whether a gene is actively used ("expressed") at any moment depends on small proteins called **transcription factors** (TFs) that physically grab onto specific patterns of DNA letters near the gene. If the right TF binds, the gene turns on. If it doesn't, the gene stays off.

**Where a TF binds is determined by the local sequence.** TFs recognize short patterns — typically 6–20 letters long — called *motifs*. A motif is fuzzy: the TF "CTCF", for example, prefers patterns that look roughly like `CCCTC` but tolerates variation. Sequence context around the motif also matters: the same motif in two different neighborhoods can attract a TF in one place and be ignored in the other.

**Why predict TF binding computationally?** Experimentally measuring where every TF binds across the genome is expensive (ChIP-seq, CUT&RUN, etc.). A model that can look at a stretch of DNA and predict "this is bound" vs. "this is not bound" lets you:
- Annotate which DNA regions are functionally important.
- Predict the effect of a mutation that changes a single letter (does it create or destroy a binding site?).
- Prioritize regions for follow-up experiments.
- Power downstream tools for drug discovery and disease genetics.

**The question this notebook answers:** *Given a ~500-letter DNA sequence, can a model predict whether a specific regulatory protein binds to it? And which kind of model does this best — a big language model, a convolutional net, a recurrent hybrid, or a classical tree-based learner?*

## How the four models think about DNA

Each model has a fundamentally different idea of what makes one DNA sequence different from another. That's the whole point of comparing them.

| Model | What it learns | Why it might win |
|---|---|---|
| **GLM (GPT)** | Pretrains on the whole human genome by predicting the next letter from previous letters. Builds a generic understanding of "what DNA looks like," then specializes. | Has seen 3 billion letters of context. Should generalize well, especially if labeled data is limited. |
| **1D CNN** | The first convolution layer's filters effectively become motif detectors (think: "is the pattern `TGACTCA` somewhere in this window?"). Higher layers combine motifs. | Motif detection is exactly what's needed for TF binding. Simple and fast. |
| **Pure BiLSTM (RNN)** | Embeds each nucleotide as a learned vector, then reads the sequence in both directions one letter at a time. Learns long-range dependencies without any built-in notion of motifs. | Pure sequence model — useful as a control to ask: "does motif detection (the CNN's bias) matter, or is general sequence modeling enough?" |
| **CNN + BiLSTM** | CNN detects motifs locally; the BiLSTM reads across the sequence in both directions to learn how motifs interact at a distance. | TFs often cooperate — two motifs spaced 50 bp apart can be the real signal. Recurrence captures that. |
| **XGBoost on k-mers** | Counts how often each short word of length k (like `ACGT`, `GGCA`, ...) appears, then learns a tree-based rule over those counts. | No deep learning, no GPU, no training time. A strong sanity check: if XGBoost already gets 85% accuracy, the deep nets need to clearly beat that to justify themselves. |

**The metrics we'll compare:**
- **Accuracy** — fraction of test sequences classified correctly.
- **F1 score** — harmonic mean of precision and recall. Important when classes are imbalanced.
- **MCC (Matthews correlation coefficient)** — balanced metric in [-1, 1]. The standard for genomic classification benchmarks because it handles imbalance well.

## Setup

In [ ]:
# If running in Colab, ensure these are installed. (Most are pre-installed.)
# %pip install -q torch numpy datasets scikit-learn xgboost matplotlib pandas biopython tqdm

import os
import math
import time
import pickle
import inspect
import gc
from dataclasses import dataclass
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset, Dataset
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Reproducibility
SEED = 1337
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Step 1 — Load the TF-binding-style dataset (shared across all 4 models)

We use the **`H3K4me3` task** from the Nucleotide Transformer benchmark (Dalla-Torre et al., InstaDeepAI). The setup:

- **Each example is a 500-letter DNA sequence.**
- **The label is binary:** `1` if the histone modification H3K4me3 is observed on this region in human cells, `0` otherwise.

*Why this counts as "TF-binding-style":* H3K4me3 is a chemical tag deposited on DNA-packaging proteins (histones) at locations where the genome is **actively being read** — i.e., where TFs and the transcription machinery are bound. Predicting H3K4me3 from raw sequence is, mechanically, the same kind of problem as predicting TF binding: take a DNA window, output a binary "bound or not." The exact same model architectures are used unchanged for true TF ChIP-seq tasks (e.g., the GUE benchmark) — just swap the dataset.

In [ ]:
# Load the binding dataset from HuggingFace.
# This downloads ~50MB and caches it.
print("Loading H3K4me3 dataset from HuggingFace...")
ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", "H3K4me3")
print(ds)

# Peek at one example
example = ds['train'][0]
print(f"\nFirst training example:")
print(f"  Sequence (first 100 bp): {example['sequence'][:100]}...")
print(f"  Sequence length: {len(example['sequence'])}")
print(f"  Label: {example['label']}  ({'bound' if example['label'] == 1 else 'not bound'})")


In [ ]:
# Extract sequences and labels into plain Python lists / numpy arrays so every model can consume them.
train_seqs_full = list(ds['train']['sequence'])
train_labels_full = np.array(ds['train']['label'])

# Carve a validation split out of train (stratified by label so both classes are represented).
train_seqs, val_seqs, train_labels, val_labels = train_test_split(
    train_seqs_full, train_labels_full,
    test_size=0.10, random_state=SEED, stratify=train_labels_full
)

# Test split is provided directly by the benchmark.
test_seqs = list(ds['test']['sequence'])
test_labels = np.array(ds['test']['label'])

# Normalize all sequences to a fixed length L by truncation / padding with 'N'.
L = max(len(s) for s in train_seqs)  # natural length in the dataset
print(f"Max sequence length: {L}")

def pad_seq(s, L):
    if len(s) >= L:
        return s[:L]
    return s + 'N' * (L - len(s))

train_seqs = [pad_seq(s.upper(), L) for s in train_seqs]
val_seqs   = [pad_seq(s.upper(), L) for s in val_seqs]
test_seqs  = [pad_seq(s.upper(), L) for s in test_seqs]

print(f"Train: {len(train_seqs)} sequences  |  pos rate: {train_labels.mean():.3f}")
print(f"Val:   {len(val_seqs)} sequences  |  pos rate: {val_labels.mean():.3f}")
print(f"Test:  {len(test_seqs)} sequences  |  pos rate: {test_labels.mean():.3f}")


In [ ]:
# Helper that all models will use to log their final test-set metrics.
results = {}  # model_name -> {'acc': ..., 'f1': ..., 'mcc': ...}

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    results[name] = {'acc': acc, 'f1': f1, 'mcc': mcc}
    print(f"[{name}]  acc={acc:.4f}  f1={f1:.4f}  mcc={mcc:.4f}")
    return acc, f1, mcc


---
# Model 1 — GLM (character-level GPT, pretrained on the human genome)

This is the same architecture as the working notebook. We will:

1. **Pretrain** a small character-level GPT on the human reference genome (`hg38`) using the standard next-letter-prediction objective. The model has to learn statistical regularities of DNA: which letters tend to follow which, where AT-rich regions live, what palindromes look like, etc.
2. **Fine-tune** that pretrained model on the H3K4me3 task by attaching a 2-class classification head on top.

The intuition for why this can work: by pretraining on all 3 billion nucleotides of human DNA, the model learns a *general* representation of DNA — what "normal" sequence looks like. Fine-tuning then teaches it which patterns specifically correlate with H3K4me3 binding. This is exactly the playbook that made BERT and GPT successful in natural language.

### 1a. Download the human reference genome (hg38) and tokenize it

In [ ]:
# WORKDIR holds the cached genome + all model checkpoints.
running_in_colab = 'COLAB_GPU' in os.environ or os.path.exists('/content')
WORKDIR = '/root/data/' if running_in_colab else os.path.expanduser('~/Downloads/gene46100-4models')
os.makedirs(WORKDIR, exist_ok=True)

fasta_file = os.path.join(WORKDIR, 'genome.fa')

if not os.path.exists(fasta_file):
    print("Downloading hg38 reference genome (~940MB compressed)...")
    if running_in_colab:
        !wget -q -O - https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz | gunzip -c > {fasta_file}
    else:
        import urllib.request, gzip, shutil
        gz = fasta_file + '.gz'
        urllib.request.urlretrieve('https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz', gz)
        with gzip.open(gz, 'rb') as fin, open(fasta_file, 'wb') as fout:
            shutil.copyfileobj(fin, fout)
        os.remove(gz)
    print("Done.")
else:
    print(f"Genome already at {fasta_file} — skipping download.")


In [ ]:
# Character-level tokenization of the genome.
# Each unique character (A, C, G, T, N, chromosome header letters, etc.) becomes one integer ID.
meta_path = os.path.join(WORKDIR, 'meta.pkl')
train_bin = os.path.join(WORKDIR, 'train.bin')
val_bin   = os.path.join(WORKDIR, 'val.bin')

if all(os.path.exists(p) for p in [meta_path, train_bin, val_bin]):
    print("Tokenized files already exist — loading vocab.")
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    vocab_size = meta['vocab_size']
    stoi, itos = meta['stoi'], meta['itos']
else:
    print("Reading genome into memory...")
    with open(fasta_file, 'r') as f:
        data = f.read()
    print(f"Genome length: {len(data):,} characters")
    chars = sorted(list(set(data)))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    print(f"Vocab size: {vocab_size}")

    # 90/10 train/val split, tokenize, save as memory-mapped binaries.
    n = len(data)
    train_data = data[: int(n * 0.9)]
    val_data   = data[int(n * 0.9) :]
    print("Tokenizing train split...")
    train_ids = np.array([stoi[c] for c in train_data], dtype=np.uint16)
    print("Tokenizing val split...")
    val_ids   = np.array([stoi[c] for c in val_data],   dtype=np.uint16)
    train_ids.tofile(train_bin)
    val_ids.tofile(val_bin)
    with open(meta_path, 'wb') as f:
        pickle.dump({'vocab_size': vocab_size, 'stoi': stoi, 'itos': itos}, f)
    del data, train_data, val_data, train_ids, val_ids
    gc.collect()
    print(f"Saved {train_bin}, {val_bin}, {meta_path}")

print(f"Final vocab_size = {vocab_size}")


### 1b. Define the GPT architecture (same as the working notebook)

In [ ]:
# Standard nanoGPT building blocks.
from torch.nn.functional import scaled_dot_product_attention

class LayerNorm(nn.Module):
    """LayerNorm with optional bias (PyTorch doesn't natively support bias=False)."""
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = scaled_dot_product_attention(
            q, k, v, attn_mask=None,
            dropout_p=self.dropout if self.training else 0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 40
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 384
    dropout: float = 0.2
    bias: bool = False


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
        n_params = sum(p.numel() for p in self.parameters()) - self.transformer.wpe.weight.numel()
        print(f"GPT params: {n_params/1e6:.2f}M")

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        return self.lm_head(x[:, [-1], :]), None

    def configure_optimizers(self, weight_decay, lr, betas, device_type):
        params = [p for p in self.parameters() if p.requires_grad]
        decay  = [p for p in params if p.dim() >= 2]
        nodecay = [p for p in params if p.dim() < 2]
        groups = [
            {'params': decay,   'weight_decay': weight_decay},
            {'params': nodecay, 'weight_decay': 0.0},
        ]
        fused_ok = 'fused' in inspect.signature(torch.optim.AdamW).parameters and device_type == 'cuda'
        return torch.optim.AdamW(groups, lr=lr, betas=betas, **({'fused': True} if fused_ok else {}))


### 1c. Pretrain the GLM on the human genome

This is the heavy step. On a Colab T4 it takes roughly **30–40 minutes** for 4,000 iterations. If you've already trained once and saved a checkpoint, the cell skips automatically.

In [ ]:
# Pretraining configuration. Same shape as the working notebook (4 layers, 4 heads, embd 384).
GLM_BLOCK_SIZE = 512  # must be >= L (500) for fine-tuning
n_layer, n_head, n_embd, dropout, bias = 4, 4, 384, 0.2, False

batch_size = 64
max_iters = 4000
eval_interval = 1000
eval_iters = 100
learning_rate = 1e-3
min_lr = 1e-4
warmup_iters = 100
lr_decay_iters = max_iters
weight_decay = 1e-1
beta1, beta2 = 0.9, 0.99
grad_clip = 1.0

# Set to True to retrain even if a checkpoint exists.
force_retrain_pretrain = False

out_dir = os.path.join(WORKDIR, 'glm_out')
os.makedirs(out_dir, exist_ok=True)
ckpt_path = os.path.join(out_dir, 'ckpt.pt')

device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype = 'bfloat16' if (device_type == 'cuda' and torch.cuda.is_bf16_supported()) else 'float32'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)


def get_batch(split, block_size):
    bin_path = train_bin if split == 'train' else val_bin
    data = np.memmap(bin_path, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


gptconf = GPTConfig(block_size=GLM_BLOCK_SIZE, vocab_size=vocab_size,
                    n_layer=n_layer, n_head=n_head, n_embd=n_embd,
                    dropout=dropout, bias=bias)
glm = GPT(gptconf).to(device)

if os.path.exists(ckpt_path) and not force_retrain_pretrain:
    print(f"Found pretraining checkpoint at {ckpt_path} — loading.")
    print("Set force_retrain_pretrain = True to retrain from scratch.")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = ckpt['model']
    for k in list(sd.keys()):
        if k.startswith('_orig_mod.'):
            sd[k[len('_orig_mod.'):]] = sd.pop(k)
    glm.load_state_dict(sd)
else:
    optimizer = glm.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)
    scaler = torch.amp.GradScaler('cuda', enabled=(dtype == 'float16' and device_type == 'cuda'))

    @torch.no_grad()
    def estimate_loss():
        out = {}
        glm.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split, GLM_BLOCK_SIZE)
                with ctx:
                    _, loss = glm(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        glm.train()
        return out

    print("Starting pretraining...")
    best_val_loss = float('inf')
    t0 = time.time()
    X, Y = get_batch('train', GLM_BLOCK_SIZE)
    for it in range(max_iters + 1):
        lr = get_lr(it)
        for g in optimizer.param_groups:
            g['lr'] = lr

        if it % eval_interval == 0:
            losses = estimate_loss()
            elapsed = time.time() - t0
            print(f"  iter {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f} | lr {lr:.2e} | {elapsed:.0f}s")
            if losses['val'] < best_val_loss and it > 0:
                best_val_loss = losses['val']
                torch.save({'model': glm.state_dict(), 'model_args': dict(
                    block_size=GLM_BLOCK_SIZE, vocab_size=vocab_size,
                    n_layer=n_layer, n_head=n_head, n_embd=n_embd,
                    dropout=dropout, bias=bias)}, ckpt_path)
                print(f"    saved checkpoint (val {best_val_loss:.4f})")

        with ctx:
            _, loss = glm(X, Y)
        X, Y = get_batch('train', GLM_BLOCK_SIZE)  # prefetch next batch
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(glm.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    print("Pretraining done.")


### 1d. Tokenize the TF-binding sequences and fine-tune the GLM

We add a tiny classification head: a single linear layer that takes the first-position output of the GPT and maps it to two class logits.

In [ ]:
# Encode TF-binding sequences using the SAME genome vocab (stoi).
# Any character not in the genome vocab (extremely rare) is mapped to 'N'.
N_id = stoi.get('N', 0)

def encode_seq(seq, max_len=L):
    return [stoi.get(c, N_id) for c in seq[:max_len]]

train_ids_clf = torch.tensor([encode_seq(s) for s in train_seqs], dtype=torch.long)
val_ids_clf   = torch.tensor([encode_seq(s) for s in val_seqs],   dtype=torch.long)
test_ids_clf  = torch.tensor([encode_seq(s) for s in test_seqs],  dtype=torch.long)
train_y = torch.tensor(train_labels, dtype=torch.long)
val_y   = torch.tensor(val_labels,   dtype=torch.long)
test_y  = torch.tensor(test_labels,  dtype=torch.long)

print(f"Tokenized: train {train_ids_clf.shape}, val {val_ids_clf.shape}, test {test_ids_clf.shape}")


class GLMClassifier(nn.Module):
    """Wrap the pretrained GLM and add a 2-class classification head."""
    def __init__(self, base_model, num_labels=2):
        super().__init__()
        self.glm = base_model
        # Take the hidden state at the first token, project to num_labels.
        self.classifier = nn.Linear(base_model.config.n_embd, num_labels)

    def forward(self, idx, labels=None):
        # Reuse the GLM's forward without targets, getting hidden states at the final ln.
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.glm.transformer.drop(self.glm.transformer.wte(idx) + self.glm.transformer.wpe(pos))
        for block in self.glm.transformer.h:
            x = block(x)
        x = self.glm.transformer.ln_f(x)            # (B, T, C)
        pooled = x.mean(dim=1)                       # mean-pool across positions for a sequence summary
        logits = self.classifier(pooled)             # (B, num_labels)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return logits, loss


glm_clf = GLMClassifier(glm).to(device)
print(f"Classifier params: {sum(p.numel() for p in glm_clf.parameters())/1e6:.2f}M")


In [ ]:
# Fine-tune loop.
glm_clf_ckpt = os.path.join(WORKDIR, 'glm_classifier.pt')
force_retrain_glm_clf = False

ft_batch_size = 32
ft_epochs = 2
ft_lr = 2e-5

train_loader = DataLoader(TensorDataset(train_ids_clf, train_y), batch_size=ft_batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(val_ids_clf,   val_y),   batch_size=ft_batch_size)
test_loader  = DataLoader(TensorDataset(test_ids_clf,  test_y),  batch_size=ft_batch_size)

if os.path.exists(glm_clf_ckpt) and not force_retrain_glm_clf:
    print(f"Loading fine-tuned GLM from {glm_clf_ckpt}")
    glm_clf.load_state_dict(torch.load(glm_clf_ckpt, map_location=device, weights_only=False))
else:
    optimizer = torch.optim.AdamW(glm_clf.parameters(), lr=ft_lr)
    glm_clf.train()
    step = 0
    for epoch in range(ft_epochs):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with ctx:
                _, loss = glm_clf(xb, labels=yb)
            loss.backward()
            optimizer.step()
            step += 1
            if step % 100 == 0:
                print(f"  epoch {epoch}  step {step}  loss {loss.item():.4f}")
    torch.save(glm_clf.state_dict(), glm_clf_ckpt)
    print(f"Saved {glm_clf_ckpt}")


# Evaluate on the held-out test set.
glm_clf.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        with ctx:
            logits, _ = glm_clf(xb)
        preds.append(logits.argmax(-1).cpu().numpy())
glm_test_pred = np.concatenate(preds)
evaluate("GLM (nanoGPT, pretrained + finetuned)", test_labels, glm_test_pred)


---
# Model 2 — 1D CNN (DeepBind / Basset style)

The CNN is the *workhorse* of regulatory-genomics ML. The trick:

- Convert each DNA sequence into a **one-hot encoding**: a 4-row matrix (rows = A/C/G/T), where each column has a `1` in the row matching that position's nucleotide.
- Slide a 1D convolution filter across this matrix. Each filter has shape `(4, k)` and learns to "fire" on a specific k-letter motif — exactly the kind of pattern TFs care about.
- Stack a couple of conv layers, max-pool to summarize over windows, then a dense head to classify.

No pretraining, no genome download. Trains in a couple of minutes.

In [ ]:
# One-hot encode all sequences. 'A'->0, 'C'->1, 'G'->2, 'T'->3, anything else -> all zeros.
BASE_TO_IDX = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

def one_hot_encode(seqs, length):
    X = np.zeros((len(seqs), 4, length), dtype=np.float32)
    for i, s in enumerate(seqs):
        for j, c in enumerate(s[:length]):
            if c in BASE_TO_IDX:
                X[i, BASE_TO_IDX[c], j] = 1.0
    return X

X_train = one_hot_encode(train_seqs, L)
X_val   = one_hot_encode(val_seqs,   L)
X_test  = one_hot_encode(test_seqs,  L)
print(f"One-hot shapes: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")


In [ ]:
class DNACnn(nn.Module):
    """Compact Basset-style CNN: 3 conv blocks + 2 FC layers."""
    def __init__(self, seq_len=L, n_classes=2):
        super().__init__()
        self.conv1 = nn.Conv1d(4,  64,  kernel_size=15, padding=7)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=11, padding=5)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=7,  padding=3)
        self.pool  = nn.MaxPool1d(kernel_size=4)
        self.drop  = nn.Dropout(0.3)
        # After 3 maxpools by factor 4: seq_len / 64
        flat = 64 * (seq_len // 64)
        self.fc1 = nn.Linear(flat, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)


cnn = DNACnn(seq_len=L).to(device)
print(f"CNN params: {sum(p.numel() for p in cnn.parameters())/1e6:.2f}M")


In [ ]:
# Train the CNN.
def train_torch_classifier(model, X_train, y_train, X_val, y_val, *,
                           epochs=8, batch_size=128, lr=1e-3, name='model'):
    train_dl = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.tensor(y_train, dtype=torch.long)),
                          batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(TensorDataset(torch.from_numpy(X_val),   torch.tensor(y_val,   dtype=torch.long)),
                          batch_size=batch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_val_mcc, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        tl = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            opt.step()
            tl += loss.item() * xb.size(0)
        tl /= len(train_dl.dataset)
        # Validation
        model.eval()
        vp = []
        with torch.no_grad():
            for xb, _ in val_dl:
                vp.append(model(xb.to(device)).argmax(-1).cpu().numpy())
        vp = np.concatenate(vp)
        vmcc = matthews_corrcoef(y_val, vp)
        print(f"  [{name}] epoch {ep+1}/{epochs}  train_loss {tl:.4f}  val_mcc {vmcc:.4f}")
        if vmcc > best_val_mcc:
            best_val_mcc = vmcc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


cnn = train_torch_classifier(cnn, X_train, train_labels, X_val, val_labels,
                              epochs=8, batch_size=128, lr=1e-3, name='CNN')

# Evaluate on test
cnn.eval()
test_dl = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.tensor(test_labels, dtype=torch.long)),
                     batch_size=256)
preds = []
with torch.no_grad():
    for xb, _ in test_dl:
        preds.append(cnn(xb.to(device)).argmax(-1).cpu().numpy())
cnn_test_pred = np.concatenate(preds)
evaluate("1D CNN (Basset-style)", test_labels, cnn_test_pred)


---
# Model 3 — Pure BiLSTM (Recurrent Neural Network)

This model is intentionally stripped down: no convolutions, no motif detection bias. Just:

- **An embedding layer** that maps each of the 5 nucleotide symbols (`A`, `C`, `G`, `T`, `N`) to a small learned vector.
- **A 2-layer bidirectional LSTM** that reads the sequence one nucleotide at a time — once left-to-right, once right-to-left — and builds up a hidden representation at every position.
- **A dense head** on the mean-pooled hidden states.

Why include this if we already have the DanQ hybrid (which uses an LSTM)? **As a control.** DanQ couples a CNN's motif-detection inductive bias with the LSTM's sequence modeling. A pure BiLSTM has *no* built-in notion of motifs — it has to learn everything from raw nucleotides. Comparing pure-RNN to CNN-only and CNN+RNN tells you:

- *If pure-RNN beats CNN-only:* sequence order and long-range context matter more than motif detection.
- *If CNN-only beats pure-RNN:* motif detection is the key inductive bias; recurrence alone struggles.
- *If CNN+RNN beats both:* the two strategies are complementary (most common outcome).

In [ ]:
# Use an integer encoding (not one-hot) so we can feed into an Embedding layer.
BASE_TO_INT = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'N': 4}

def int_encode(seqs, length):
    X = np.full((len(seqs), length), BASE_TO_INT['N'], dtype=np.int64)
    for i, s in enumerate(seqs):
        for j, c in enumerate(s[:length]):
            X[i, j] = BASE_TO_INT.get(c, BASE_TO_INT['N'])
    return X

Xi_train = int_encode(train_seqs, L)
Xi_val   = int_encode(val_seqs,   L)
Xi_test  = int_encode(test_seqs,  L)
print(f"Integer-encoded shapes: train {Xi_train.shape}, val {Xi_val.shape}, test {Xi_test.shape}")


In [ ]:
class DNABiLSTM(nn.Module):
    """Pure RNN: embed nucleotides, then a 2-layer BiLSTM, then a small FC head."""
    def __init__(self, vocab=5, emb_dim=32, hidden=128, n_layers=2, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb_dim)
        self.lstm  = nn.LSTM(input_size=emb_dim, hidden_size=hidden,
                              num_layers=n_layers, batch_first=True,
                              bidirectional=True, dropout=0.3 if n_layers > 1 else 0.0)
        self.drop  = nn.Dropout(0.4)
        self.fc1   = nn.Linear(2 * hidden, 128)
        self.fc2   = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (B, L)  int tokens
        e = self.embed(x)             # (B, L, emb_dim)
        h, _ = self.lstm(e)           # (B, L, 2*hidden)
        pooled = h.mean(dim=1)        # mean over the sequence
        z = self.drop(F.relu(self.fc1(pooled)))
        return self.fc2(z)


bilstm = DNABiLSTM().to(device)
print(f"BiLSTM params: {sum(p.numel() for p in bilstm.parameters())/1e6:.2f}M")


In [ ]:
# Train the pure BiLSTM. Note: input dtype is int64 (token ids), not float.
def train_torch_classifier_int(model, X_train, y_train, X_val, y_val, *,
                                epochs=8, batch_size=128, lr=1e-3, name='model'):
    train_dl = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.tensor(y_train, dtype=torch.long)),
                          batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(TensorDataset(torch.from_numpy(X_val),   torch.tensor(y_val,   dtype=torch.long)),
                          batch_size=batch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_val_mcc, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        tl = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # RNNs need gradient clipping
            opt.step()
            tl += loss.item() * xb.size(0)
        tl /= len(train_dl.dataset)
        model.eval()
        vp = []
        with torch.no_grad():
            for xb, _ in val_dl:
                vp.append(model(xb.to(device)).argmax(-1).cpu().numpy())
        vp = np.concatenate(vp)
        vmcc = matthews_corrcoef(y_val, vp)
        print(f"  [{name}] epoch {ep+1}/{epochs}  train_loss {tl:.4f}  val_mcc {vmcc:.4f}")
        if vmcc > best_val_mcc:
            best_val_mcc = vmcc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


bilstm = train_torch_classifier_int(bilstm, Xi_train, train_labels, Xi_val, val_labels,
                                     epochs=8, batch_size=128, lr=1e-3, name='BiLSTM')

# Evaluate on test
bilstm.eval()
test_dl_int = DataLoader(TensorDataset(torch.from_numpy(Xi_test), torch.tensor(test_labels, dtype=torch.long)),
                         batch_size=256)
preds = []
with torch.no_grad():
    for xb, _ in test_dl_int:
        preds.append(bilstm(xb.to(device)).argmax(-1).cpu().numpy())
bilstm_test_pred = np.concatenate(preds)
evaluate("Pure BiLSTM (RNN)", test_labels, bilstm_test_pred)


---
# Model 4 — CNN + BiLSTM (DanQ-style hybrid)

DanQ (Quang & Xie, 2016) is one of the most cited models in regulatory genomics. The architecture:

- **First, a CNN layer** extracts local motifs from one-hot DNA — same idea as Model 2's first layer.
- **Then, a bidirectional LSTM** reads across the convolved feature map from left-to-right AND right-to-left, learning how motifs at different positions interact. This is what plain CNNs struggle with: a CNN sees motifs but doesn't easily learn that "motif X *50 bp upstream of* motif Y" is what matters.
- **A dense head** finishes the job.

In [ ]:
class DanQ(nn.Module):
    """DanQ-style: Conv -> MaxPool -> BiLSTM -> FC."""
    def __init__(self, seq_len=L, n_classes=2, n_filters=128, kernel=15, lstm_hidden=64):
        super().__init__()
        self.conv = nn.Conv1d(4, n_filters, kernel_size=kernel, padding=kernel//2)
        self.pool = nn.MaxPool1d(kernel_size=8)
        self.drop1 = nn.Dropout(0.2)
        self.lstm = nn.LSTM(input_size=n_filters, hidden_size=lstm_hidden,
                            batch_first=True, bidirectional=True)
        self.drop2 = nn.Dropout(0.4)
        # After pooling by 8: seq_len // 8 timesteps, each with 2*lstm_hidden features
        flat = (seq_len // 8) * 2 * lstm_hidden
        self.fc1 = nn.Linear(flat, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (B, 4, L)
        x = self.pool(F.relu(self.conv(x)))   # (B, F, L/8)
        x = self.drop1(x)
        x = x.transpose(1, 2)                 # (B, L/8, F)
        x, _ = self.lstm(x)                   # (B, L/8, 2*hidden)
        x = self.drop2(x.flatten(1))
        x = F.relu(self.fc1(x))
        return self.fc2(x)


danq = DanQ(seq_len=L).to(device)
print(f"DanQ params: {sum(p.numel() for p in danq.parameters())/1e6:.2f}M")


In [ ]:
danq = train_torch_classifier(danq, X_train, train_labels, X_val, val_labels,
                                epochs=8, batch_size=128, lr=1e-3, name='DanQ')

# Evaluate on test
danq.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_dl:
        preds.append(danq(xb.to(device)).argmax(-1).cpu().numpy())
danq_test_pred = np.concatenate(preds)
evaluate("CNN+BiLSTM (DanQ-style)", test_labels, danq_test_pred)


---
# Model 5 — XGBoost on k-mer features (classical ML baseline)

A k-mer is just a short DNA word of length k: `AAAA`, `AAAC`, `AAAG`, ... For DNA there are 4 letters, so 4^k possible k-mers. For each sequence, we count how often each k-mer appears and normalize by sequence length. That gives a fixed-length feature vector.

Why bother with this in a deep-learning notebook? Two reasons:

1. **It is shockingly competitive.** On many TF-binding tasks, gradient-boosted trees on k-mers come within a few points of neural models. If your fancy model isn't clearly beating XGBoost, the fancy model isn't actually learning anything useful.
2. **It's interpretable.** XGBoost can tell you exactly which k-mers drive its predictions — i.e., which short DNA words are most associated with binding. This is much harder to do for neural models.

In [ ]:
# Install xgboost if not available
try:
    import xgboost as xgb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'], check=True)
    import xgboost as xgb

from itertools import product

K = 4  # k-mer length; 4^4 = 256 features. Try K=5 or 6 for stronger model + more features.
KMERS = [''.join(p) for p in product('ACGT', repeat=K)]
KMER_INDEX = {km: i for i, km in enumerate(KMERS)}
print(f"Using k = {K}  ({len(KMERS)} possible k-mers)")

def kmer_features(seqs, k=K):
    X = np.zeros((len(seqs), len(KMERS)), dtype=np.float32)
    for i, s in enumerate(seqs):
        s = s.upper()
        total = 0
        for j in range(len(s) - k + 1):
            km = s[j:j+k]
            idx = KMER_INDEX.get(km)
            if idx is not None:
                X[i, idx] += 1
                total += 1
        if total > 0:
            X[i] /= total  # normalize to frequencies
    return X

print("Extracting k-mer features...")
Xk_train = kmer_features(train_seqs)
Xk_val   = kmer_features(val_seqs)
Xk_test  = kmer_features(test_seqs)
print(f"Shapes: train {Xk_train.shape}, val {Xk_val.shape}, test {Xk_test.shape}")


In [ ]:
# Train XGBoost. Fast on CPU; switch tree_method='hist' for speed.
xgb_clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1,
    random_state=SEED,
)
xgb_clf.fit(
    Xk_train, train_labels,
    eval_set=[(Xk_val, val_labels)],
    verbose=False,
)
xgb_test_pred = xgb_clf.predict(Xk_test)
evaluate("XGBoost on k-mers", test_labels, xgb_test_pred)


In [ ]:
# Show the top-15 most predictive k-mers — XGBoost gives this for free.
importance = xgb_clf.feature_importances_
top = np.argsort(importance)[::-1][:15]
print("Most predictive k-mers (by XGBoost gain):")
for rank, idx in enumerate(top, 1):
    print(f"  {rank:2d}. {KMERS[idx]}  (importance={importance[idx]:.4f})")


---
# Results — head-to-head comparison

All five models were trained and evaluated on the **same train / val / test split** of the H3K4me3 dataset. Below is the side-by-side comparison.

In [ ]:
results_df = (pd.DataFrame(results).T
               .rename_axis('Model').reset_index()
               .sort_values('mcc', ascending=False)
               .reset_index(drop=True))
results_df[['acc', 'f1', 'mcc']] = results_df[['acc', 'f1', 'mcc']].round(4)
print(results_df.to_string(index=False))


In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['acc', 'f1', 'mcc']
x = np.arange(len(results_df))
width = 0.25
for i, m in enumerate(metrics):
    ax.bar(x + (i - 1) * width, results_df[m], width, label=m.upper())
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right')
ax.set_ylabel("Score")
ax.set_title("Test-set performance on H3K4me3 prediction")
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## How to read these results

A few things to look for in your specific run (numbers will vary slightly from run to run because of random initialization, but the overall ranking is usually stable):

**The GLM does NOT always win.** This is the most common surprise for newcomers. A character-level GPT pretrained on the whole genome has a *lot* of capacity and prior knowledge, but for a 500 bp binary classification task, simpler models often perform comparably or better — especially when the training set is small enough that the GLM hasn't fully specialized to the task during fine-tuning.

**XGBoost is usually within a few points of the neural models.** Don't dismiss it. For many regulatory genomics tasks, k-mer frequency is most of the signal. If your deep model isn't clearly beating XGBoost, it isn't really learning sequence structure — it's just learning composition.

**The pure BiLSTM is the most informative model in this lineup.** Compare it to the CNN and to DanQ:

- *If BiLSTM ≥ CNN:* sequence context and order are doing the heavy lifting — motif detection alone isn't enough.
- *If BiLSTM < CNN:* this task is largely about which short motifs are present, not how they interact. Local pattern matching wins.
- *If DanQ > both BiLSTM and CNN:* the two strategies combine well — this is the most common outcome on TF-binding-style problems.

**DanQ (CNN + BiLSTM) typically beats both the plain CNN and the pure BiLSTM by a small margin.** The CNN's motif inductive bias plus the LSTM's long-range modeling cover each other's blind spots.

**Tradeoffs to consider:**

- **Compute cost:** GLM (pretraining) >> DanQ > BiLSTM ≈ CNN >> XGBoost. The GLM pretraining alone dwarfs everything else combined. Pure RNNs are roughly as expensive to train as DanQ but converge more slowly.
- **Data efficiency:** GLM should win when labeled data is *very* limited (since pretraining transfers). With ~30k labeled examples like H3K4me3, the gap shrinks.
- **Interpretability:** XGBoost ≫ CNN > DanQ > BiLSTM > GLM. XGBoost tells you which k-mers matter; CNNs let you visualize learned motif filters; recurrent and transformer models are mostly opaque.
- **Adaptability to longer sequences:** CNN, DanQ, BiLSTM scale linearly with sequence length. The GLM has quadratic attention (problematic past 1–2 kb). XGBoost is invariant to length.

## Where to go next

- **Swap the dataset.** This same notebook works unchanged on true TF binding (GUE benchmark, `leannmlindsey/GUE` config `tf/*`), enhancer classification, splice sites, etc.
- **Pretrain the GLM longer.** 4,000 iterations is a tiny budget. The original nanoGPT paper trains for 600,000.
- **Try a bidirectional pretraining objective.** Replace the causal GPT with a BERT-style masked language model (DNABERT). Bidirectional context generally outperforms causal on classification.
- **Try a GRU instead of LSTM.** GRUs are smaller and often comparable in quality — a useful sanity check for the recurrent models.
- **Ensemble the five models.** Average their test predictions. This almost always beats any single model.